# DefectLens — run the whole pipeline in Colab

AI ARENA 2026 · DefectLens track. This notebook downloads our code and the organisers' practice data, shows the shift simulator, trains a model, stress-tests it against a naive baseline, writes a CSV in the organisers' format, and draws Grad-CAM heat-maps.

**First:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all (about 6–8 minutes).

The submitted models were trained the same way for 30 epochs, three times with different seeds; here we use 10 epochs to keep it quick.

## 1 · Code, data and pretrained weights

In [ ]:
!git clone -q https://github.com/DAYAANIDHISV/defectlens.git
%cd defectlens
!mkdir -p models data
# ResNet18 pretrained on ImageNet (the starting point we fine-tune)
!curl -sL -o models/resnet18-imagenet.pth https://download.pytorch.org/models/resnet18-f37072fd.pth
# The organisers' participant package, from their own repository
!curl -sL -o /tmp/arena.zip https://github.com/sanjai-umashankar/AI-Arena-AIML-Hackathon-2026/raw/main/AI_ARENA_PARTICIPANT.zip
!unzip -q -o /tmp/arena.zip -d /tmp/arena && cp -r /tmp/arena/AI_ARENA_PARTICIPANT/PARTICIPANT_PACKAGE/DefectLens data/
!echo "training photos: $(find data/DefectLens/train -name '*.png' | wc -l)   validation photos: $(find data/DefectLens/validation -name '*.png' | wc -l)"

## 2 · The shift simulator
Every training photo is randomly turned, relit, re-backgrounded, covered with glare and degraded like a poor camera — the defect is the only thing that never changes. One part per class, under each disturbance:

In [ ]:
!python preview_shifts.py
from IPython.display import Image, display
display(Image('docs/shift_examples.png'))

## 3 · Train
Our recipe (`train.py`): ResNet18, AdamW, label smoothing 0.1, one-cycle learning rate, every photo disturbed on the fly. Then the naive baseline: the same network with **no** disturbances (`--plain`).

In [ ]:
!python train.py --name colab --epochs 10 2>&1 | grep -v Warning

In [ ]:
!python train.py --name colab-plain --plain --epochs 10 2>&1 | grep -v Warning

## 4 · Stress test: naive vs ours
The 200 validation photos redrawn under 29 changed conditions, each scored separately (macro-F1). Validation alone is too easy to tell the two apart — this is where they differ.

In [ ]:
!python evaluate.py colab-plain colab --out colab 2>&1 | grep -v Warning
from IPython.display import Markdown
display(Markdown(open('outputs/stress_table_colab.md').read()))

## 5 · Predict a folder → CSV in the organisers' format
(`sample_id,predicted_class,confidence`; eight turned and mirrored views, each also zoomed, averaged per photo.)

In [ ]:
!python predict.py data/DefectLens/validation --models colab --out submission_colab.csv 2>&1 | grep -v Warning
!head -6 submission_colab.csv

## 6 · Where the model looked (Grad-CAM)
Photo, then heat-map, two per class — it should glow on the defect, not the background.

In [ ]:
!python show_heatmaps.py colab
display(Image('docs/heatmaps_colab.png'))